# Setup

In [96]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# use spark (distributed MapReduce) or pandas (local computation)
LOCAL_COMPUTATION = False

# spark configuration
os.environ["JAVA_HOME"] = "" or "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "" or "/content/spark"

# subsampling: percentage of commenting users to keep
USERS_PERCENTAGE = 0.1
assert 0 < USERS_PERCENTAGE <= 1, "invalid USER_PERCENTAGE, should be in (0, 1]"

In [97]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)

Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [98]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j

Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [99]:
# setup pandas (tests before scaling to spark)
%pip install pandas
import pandas as pd
articles = pd.read_csv("dataset/nyt-articles-2020.csv")
comments = pd.read_csv("dataset/nyt-comments-part0.csv") # TODO: this is already a sample
print(f"Dataset size: {len(articles)=}, {len(comments)=}")
(articles.columns, comments.columns)

Note: you may need to restart the kernel to use updated packages.
Dataset size: len(articles)=16787, len(comments)=500000


(Index(['newsdesk', 'section', 'subsection', 'material', 'headline', 'abstract',
        'keywords', 'word_count', 'pub_date', 'n_comments', 'uniqueID'],
       dtype='object'),
 Index(['commentID', 'status', 'commentSequence', 'userID', 'userDisplayName',
        'userLocation', 'userTitle', 'commentBody', 'createDate', 'updateDate',
        'approveDate', 'recommendations', 'replyCount', 'editorsSelection',
        'parentID', 'parentUserDisplayName', 'depth', 'commentType', 'trusted',
        'recommendedFlag', 'permID', 'isAnonymous', 'articleID'],
       dtype='object'))

In [100]:
# subsampling
# WARNING: executing this cell multiple times without executing previous cell will continue to shrink the dataset
if USERS_PERCENTAGE < 1:
    total_users = len(comments['userID'].unique())
    subsampled_users = int(total_users * USERS_PERCENTAGE)
    top_users = comments['userID'].value_counts().nlargest(subsampled_users).index # keep top users
    print(top_users) # TODO

    print(f"Subsampling from {total_users} to {subsampled_users} users:")

    print(f"- comments: {len(comments)} -> ", end="")
    comments = comments[comments['userID'].isin(top_users)].copy() # keep only comments from these users
    print(f"{len(comments)}")

    active_article_ids = comments['articleID'].unique() # keep only commented articles
    print(f"- articles: {len(articles)} -> ", end="")
    articles = articles[articles['uniqueID'].isin(active_article_ids)].copy() # keep only commented articles
    print(f"{len(articles)}")

Index([ 67892453,  68938663,  73444633,  33668404,  69498476,  78358680,
        63483890,  72967915,  87081404,  93082167,
       ...
        49632588,     61360,  80245071,  22822988,  28571829,  21227474,
        65684325, 103084739,  78118899,  42471841],
      dtype='int64', name='userID', length=9143)
Subsampling from 91437 to 9143 users:
- comments: 500000 -> 313145
- articles: 16787 -> 1593


# Utility Matrix

In [101]:
utility_matrix = pd.crosstab(index=comments['userID'], columns=comments['articleID'])
utility_matrix = (utility_matrix > 0).astype(int) # ignore multiple comments on the same article
utility_matrix

articleID,nyt://article/00036db7-8494-5141-b0ac-414118caabee,nyt://article/00168469-d42e-5a6f-9c81-90e3306c6c85,nyt://article/004716c5-8943-5bf4-8293-864ab083d676,nyt://article/00573cb4-92bb-53b7-ad9a-6d4d1e734704,nyt://article/00bea7d5-c04a-509d-a82f-7537a16dfeb1,nyt://article/012aeb93-8149-522b-a18b-d6cfc620fb22,nyt://article/016cf3b3-56e9-53e4-882c-549209211afb,nyt://article/0178a7cd-b6b8-58b2-afe4-3d1f68aef248,nyt://article/01a433cd-37ce-5e77-a39b-fe4af162a34f,nyt://article/01b7c5aa-9cf5-56a0-8162-3b8d606ce533,...,nyt://interactive/c46aa52e-ec78-5e20-8145-acaa9d1d5705,nyt://interactive/c696fd2f-2287-562e-91ec-3f07747a9923,nyt://interactive/c6ec5381-c3f8-5a71-b23a-a0d9b18c7237,nyt://interactive/c891ed57-1184-52d9-bd9c-cf9f7bf23f13,nyt://interactive/d0598a7b-06ba-56db-9df4-7a2afa1c95e3,nyt://interactive/dcc11686-2ae2-562c-9e2b-738e9140b9d4,nyt://interactive/df6cd908-52b0-5851-ac81-e7b8039d50d1,nyt://interactive/e31c6ca3-4bea-5a56-b358-7986e7590f1d,nyt://interactive/e5bd1257-4e08-5656-bc03-091a2597e336,nyt://interactive/f91ad36a-9e5c-5e95-a210-37f1b797fdf3
userID,,,,,,,,,,,,,,,,,,,,,
2593,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8010,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8638,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13323,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
17282,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109587542,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109738098,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Content-Based Recommendations

### Articles Profile

In [102]:
# items profile
feature_columns = ["newsdesk", "section", "subsection"]
item_features_matrix = pd.get_dummies(articles[feature_columns], dtype=int)
item_features_matrix.index = articles['uniqueID'].values
# articles["profile"] = item_features_matrix.values.tolist()
item_features_matrix

,newsdesk_Arts,newsdesk_Arts&Leisure,newsdesk_BookReview,newsdesk_Books,newsdesk_Business,newsdesk_Climate,newsdesk_Culture,newsdesk_Dining,newsdesk_Editorial,newsdesk_Express,...,subsection_Pro Football,subsection_Self-Care,subsection_Skiing,subsection_Soccer,subsection_Sunday Review,subsection_Television,subsection_Tennis,subsection_The Daily,subsection_Weddings,"subsection_Wine, Beer & Cocktails"
nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/04bc90f0-b20b-511c-b5bb-3ce13194163f,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/bd8647b3-8ec6-50aa-95cf-2b81ed12d2dd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/ac742403-9ccd-522f-9a1e-a90feb6c0acd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nyt://article/b9c293a0-dfb4-5630-9948-8a39be8afce8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/203dd447-5d08-5745-95b8-b97de174d4b9,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
nyt://article/1e2029bd-8b3d-5e1e-9e78-28e5ff746998,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
nyt://article/697c75a8-2600-5e81-98ac-38193d89ae12,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


### Users Profile

In [103]:
# user profile: count how many times each user commented on articles with each label
user_profiles = []

for user_id in utility_matrix.index:
    commented_articles = utility_matrix.loc[user_id][utility_matrix.loc[user_id] == 1].index # articles commented by this user
    user_profile = item_features_matrix.loc[item_features_matrix.index.isin(commented_articles)].sum(axis=0) # sum features vectors for these articles
    user_profiles.append(user_profile)

user_features_matrix = pd.DataFrame(user_profiles, index=utility_matrix.index)
user_features_matrix

,newsdesk_Arts,newsdesk_Arts&Leisure,newsdesk_BookReview,newsdesk_Books,newsdesk_Business,newsdesk_Climate,newsdesk_Culture,newsdesk_Dining,newsdesk_Editorial,newsdesk_Express,...,subsection_Pro Football,subsection_Self-Care,subsection_Skiing,subsection_Soccer,subsection_Sunday Review,subsection_Television,subsection_Tennis,subsection_The Daily,subsection_Weddings,"subsection_Wine, Beer & Cocktails"
userID,,,,,,,,,,,,,,,,,,,,,
2593,0,0,0,0,2,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
8010,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8638,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
13323,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
17282,0,2,0,0,3,0,1,0,2,0,...,2,0,0,0,3,2,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109587542,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109738098,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Calculate Recommendations

In [104]:
from sklearn.metrics.pairwise import cosine_similarity # TODO: implement from scratch?
similarity_array = cosine_similarity(user_features_matrix, item_features_matrix)
similarity_df = pd.DataFrame(
    similarity_array,
    index=user_features_matrix.index,
    columns=item_features_matrix.index
)
similarity_df.to_csv("similarity_df.csv")
similarity_df

,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd,nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c,nyt://article/04bc90f0-b20b-511c-b5bb-3ce13194163f,nyt://article/bd8647b3-8ec6-50aa-95cf-2b81ed12d2dd,nyt://article/ac742403-9ccd-522f-9a1e-a90feb6c0acd,nyt://article/dea10063-7440-586f-bc32-db1dbaa9893f,nyt://article/dfd5bb59-f330-5c01-ae02-221e6f9cae7a,nyt://article/6227c991-6a3d-5eb1-92c9-a618f6e20dbd,nyt://article/7d9a4f78-44ef-5756-bc2b-4c13cbfa0d5a,nyt://article/da74e3b3-f027-5b03-9052-0bc2e69a2f3e,...,nyt://article/23ec0066-2831-599d-bbff-cef34498ab07,nyt://article/2ae8b3b1-f038-5953-a83a-8b75389010da,nyt://article/67b90b5b-fbb1-5f4b-ab12-7fcc24793bcf,nyt://article/d118dda5-b976-56f9-81c7-a961dcff8e84,nyt://article/46182ac4-985b-5b2d-9c4c-5ca4c9552fdd,nyt://article/b9c293a0-dfb4-5630-9948-8a39be8afce8,nyt://article/203dd447-5d08-5745-95b8-b97de174d4b9,nyt://article/1e2029bd-8b3d-5e1e-9e78-28e5ff746998,nyt://article/697c75a8-2600-5e81-98ac-38193d89ae12,nyt://article/49cedd40-d966-5fa1-a04a-1db9abc2756a
userID,,,,,,,,,,,,,,,,,,,,,
2593,0.185341,0.0,0.030890,0.030890,0.680985,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.226995,0.370681,0.370681,0.000000,0.000000,0.302660,0.403547
8010,0.046424,0.0,0.000000,0.000000,0.000000,0.278543,0.278543,0.278543,0.000000,0.000000,...,0.00000,0.0,0.000000,0.644383,0.092848,0.092848,0.000000,0.000000,0.075810,0.947623
8638,0.396059,0.0,0.099015,0.099015,0.080845,0.000000,0.000000,0.000000,0.099015,0.161690,...,0.19803,0.0,0.099015,0.242536,0.693103,0.693103,0.000000,0.000000,0.565916,0.161690
13323,0.553519,0.0,0.000000,0.000000,0.150649,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.075324,0.968658,0.968658,0.000000,0.000000,0.790906,0.150649
17282,0.552536,0.0,0.000000,0.000000,0.000000,0.157867,0.157867,0.157867,0.039467,0.000000,...,0.00000,0.0,0.118401,0.193347,0.868271,0.868271,0.128898,0.193347,0.805614,0.128898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0.000000,0.0,0.250000,0.250000,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
109587542,0.267261,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.145479,...,0.00000,0.0,0.000000,0.727393,0.534522,0.534522,0.000000,0.000000,0.436436,0.727393
109738098,0.500000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.816497,0.000000


In [105]:
articles_lookup = articles.set_index("uniqueID", drop=False)

def print_article(i, article_id, score=None):
    if article_id in articles_lookup.index:
        row = articles_lookup.loc[article_id]
        title = row.get("headline", row.get("title", "N/A"))
        abstract = row.get("abstract", "N/A")
        newsdesk = row.get("newsdesk", "N/A")
        section = row.get("section", "N/A")
        subsection = row.get("subsection", "N/A")
    else:
        title = abstract = newsdesk = section = subsection = "N/A"

    score_str = f" | score: {score:.5f}" if score is not None else ""
    print(f"{i}. article_id: {article_id}{score_str} | {title} | {abstract} | {newsdesk=} | {section=} | {subsection=}")

def print_results(user_id, recommendations=5):

    commented_ids = []
    if user_id in utility_matrix.index:
        commented = utility_matrix.loc[user_id]
        commented_ids = commented[commented > 0].index.tolist()

    scores = similarity_df.loc[user_id].sort_values(ascending=False)
    # remove alreayd commented articles
    scores = scores.drop(index=commented_ids, errors="ignore")


    print(f"=== Top {len(scores.head(recommendations))} recommendations for user {user_id}:")
    for i, (article_id, score) in enumerate(scores.head(recommendations).items(), start=1):
        print_article(i, article_id, score)

    print(f"\n=== Articles already commented by user {user_id} ({len(commented_ids)}):")
    for i, article_id in enumerate(commented_ids, start=1):
        print_article(i, article_id)

print_results(similarity_df.index[0], 10)


=== Top 10 recommendations for user 2593:
1. article_id: nyt://article/559b427e-a5af-5a06-b668-bd83670e1ca3 | score: 0.75665 | China Pledged to Build a New Hospital in 10 Days. It’s Close. | State news outlets reported that the 1,000-bed facility would accept patients from Monday even as construction workers raced to complete it. | newsdesk='Foreign' | section='World' | subsection='Asia Pacific'
2. article_id: nyt://article/643a5a43-56e8-5f61-8f30-e79268d7b2e2 | score: 0.75665 | In Blow to Beijing, Taiwan Re-elects Tsai Ing-wen as President | The victory was a remarkable comeback for Ms. Tsai and suggested that Beijing’s pressure campaign had backfired. | newsdesk='Foreign' | section='World' | subsection='Asia Pacific'
3. article_id: nyt://article/89f51792-75fe-560f-a798-031c32fdd212 | score: 0.75665 | As Coronavirus Spreads, So Does Anti-Chinese Sentiment | Fears of the outbreak have fueled xenophobia as a wave of panic spreads, sometimes outstripping practical concerns. | newsdesk='F